## 第7章　矩阵的逆与线性方程组 —— 解方程的艺术

> 本章目标：理解逆矩阵 = "撤销变换"的操作。掌握用 `np.linalg.inv` 解线性方程组，理解奇异矩阵（不可逆）的本质——变换把空间"压扁"了，信息不可恢复。最终落地到 AI 经典公式：线性回归闭式解 **w** = (**X**ᵀ**X**)⁻¹**X**ᵀ**y**。
> 前置知识：第 6 章（矩阵乘法、空间变换）、第 4 章（向量运算）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 7.1　单位矩阵 —— "什么都不做"的变换

在第 6 章你看到矩阵能把空间旋转、拉伸、剪切。那有没有一个矩阵，对任何向量乘上去之后——什么都不变？有的：**单位矩阵 I（Identity Matrix）**。

$$**I** = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

主对角线全 1，其余全 0。对任何向量 **x**：**I** **x** = **x**。对任何矩阵 **A**：**A I** = **I A** = **A**。单位矩阵是矩阵世界里的"数字 1"——乘以它等于没乘。

📐 **定义　单位矩阵（Identity Matrix）**：n×n 方阵，对角线为 1，其余为 0。`np.eye(n)` 生成。**A I** = **I A** = **A**（前提是形状兼容）。

💻 **代码　单位矩阵：验证"等于没乘"**


In [ ]:
import numpy as np

I3 = np.eye(3)
print(f"3×3 单位矩阵:\n{I3}")

A = np.array([[2.0, 1.0, 3.0],
              [1.0, 4.0, 0.0],
              [5.0, 2.0, 1.0]])
x = np.array([3.0, 1.0, 2.0])

print(f"\nI @ x = {I3 @ x}  ← 等于 x ✓")
print(f"A @ I = ...\n{A @ I3}  ← 等于 A ✓")
print(f"I @ A = ...\n{I3 @ A}  ← 等于 A ✓")


> **关键洞察**：单位矩阵看起来"没用"，但它定义了逆矩阵的参照系——**A⁻¹ 就是那个"乘了之后变回 I"的矩阵。**

🔗 **AI 连接**：PyTorch 权重初始化（第 25 章）有时将 RNN 的循环权重初始化为单位矩阵——确保在训练初期，信息能"无损地"通过时间步。

---


### 7.2　逆矩阵 —— "撤销变换"

如果你用矩阵 **A** 把向量 **x** 变换成 **y** = **A****x**，逆矩阵 **A**⁻¹ 可以从 **y** 恢复出 **x**：**x** = **A**⁻¹**y**。用大白话说：**A 把空间扭成某种形状，A⁻¹ 把它扭回来。**

但并非所有矩阵都有逆矩阵。条件：**A 必须是方阵（行数=列数），且行列式 ≠ 0（变换没有把空间"压扁"）。** 压扁意味着变换丢失了维度信息——一旦丢失，就永远找不回来了。

📐 **定义　逆矩阵（Inverse Matrix）**：若存在 **A**⁻¹ 使得 **A A**⁻¹ = **A**⁻¹**A** = **I**，则称 **A** 可逆（invertible）。`np.linalg.inv(A)` 计算。若 **A** 不可逆（奇异矩阵 / singular），`inv` 会报错，此时可用 `np.linalg.pinv`（伪逆 / 最小二乘近似）。

💻 **代码　逆矩阵：验证 "变换再逆变换 = 什么都没发生"**


In [ ]:
import numpy as np

# 可逆矩阵
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])
A_inv = np.linalg.inv(A)

print(f"A =\n{A}")
print(f"A⁻¹ =\n{A_inv.round(4)}")
print(f"\nA @ A⁻¹ =\n{A @ A_inv.round(6)}  ← 应该是单位矩阵")
print(f"A⁻¹ @ A =\n{A_inv @ A.round(6)}  ← 也应该是单位矩阵")

# 验证：变换后能恢复
x = np.array([5.0, 2.0])
y = A @ x          # A 变换
x_recovered = A_inv @ y  # A⁻¹ 逆变换
print(f"\n原向量 x = {x}")
print(f"A @ x = {y}")
print(f"A⁻¹ @ (A @ x) = {x_recovered} ← 完美恢复 ✓")

# 奇异矩阵示例——行列式为 0
B = np.array([[1.0, 2.0],
              [2.0, 4.0]])  # 第二行 = 第一行 × 2 → 行列式为 0
print(f"\n奇异矩阵 B:\n{B}")
print(f"B 的行列式 = {np.linalg.det(B):.0f} ← 为 0，不可逆")
try:
    np.linalg.inv(B)
except np.linalg.LinAlgError:
    print("np.linalg.inv(B) 报错: 奇异矩阵不可逆！")


> **关键洞察**：奇异矩阵 = 变换把空间"压扁"了。B = [[1,2],[2,4]] 的两行共线，意味着它把整个 2D 平面压缩到了 1 条直线上——一旦压缩，就无法从 1 条线恢复原来的 2D 平面。行列式 = 0 就是"面积/体积被压到 0"的数学表达。

🔗 **AI 连接**：第 24 章 Adam 优化器中，`v_hat / (sqrt(s_hat) + eps)` 的除法在数学上等价于求"对角矩阵的逆"——每个参数根据自身梯度历史独立缩放。第 26 章线性回归闭式解的核心就是 `(XᵀX)⁻¹`——对协方差矩阵求逆。

---


### 7.3　用逆矩阵解方程组 —— 从"猜"到"秒算"

方程组：
```
2x + 3y = 8
5x + 4y = 13
```

写成矩阵形式：**A****w** = **b**，其中 **A** = [[2,3],[5,4]]，**w** = [x,y]ᵀ，**b** = [8,13]ᵀ。两边同乘 **A**⁻¹：**w** = **A**⁻¹**b**。**不需要消元、不需要迭代——一次矩阵运算直接出答案。**

这就是"闭式解"（Closed-Form Solution）的含义：不靠反复试探，一步到位算出精确答案。

💻 **代码　矩阵方式解二元一次方程组**


In [ ]:
import numpy as np

# 方程组: 2x + 3y = 8,  5x + 4y = 13
A = np.array([[2.0, 3.0],
              [5.0, 4.0]])
b = np.array([8.0, 13.0])

# 闭式解
w = np.linalg.inv(A) @ b  # w = A⁻¹ b

print(f"方程组的解: x = {w[0]:.1f}, y = {w[1]:.1f}")
# 验证
print(f"2×{w[0]:.1f} + 3×{w[1]:.1f} = {2*w[0]+3*w[1]:.1f} (应为 8) ✓")
print(f"5×{w[0]:.1f} + 4×{w[1]:.1f} = {5*w[0]+4*w[1]:.1f} (应为 13) ✓")

# 对比：np.linalg.solve 专门解方程组——内部做了数值优化
w_solve = np.linalg.solve(A, b)
print(f"\nnp.linalg.solve 结果: {w_solve} ← 与 inv(A) @ b 一致")


> **关键洞察**：`A⁻¹ @ b` 和 `np.linalg.solve(A, b)` 结果一致，但 solve 更快更稳定——它不显式计算逆矩阵，而是用 LU 分解直接解方程。但 `inv(A) @ b` 的写法更直观地揭示了数学含义。第 26 章的线性回归闭式解就长这样。

---


### 7.4　伪逆 —— 当精确解不存在时的"最佳近似"

不是所有方程组都有精确解。**X****w** = **y** 中如果方程数量多于未知数（样本数 > 特征数），通常没有恰好满足所有方程的 **w**——这时需要找"让所有方程尽可能接近成立"的那个 **w**。这就是**最小二乘解**：min ‖**X****w** − **y**‖²。

伪逆（Pseudoinverse / Moore-Penrose Inverse）`np.linalg.pinv(X)` 就是做这件事的——即使 **X** 不是方阵、即使 **X** 不可逆，伪逆也能给出最小二乘意义下的"最佳近似解"：**w** = pinv(**X**) @ **y**。

📐 **定义　伪逆（Pseudoinverse）**：对任意 m×n 矩阵 X，pinv(X) 给出使得 ‖Xw−y‖² 最小的 w。当 X 可逆时 pinv(X) = inv(X)，当 X 奇异或非方阵时给出最小二乘解。

💻 **代码　伪逆 vs 精确逆：多方程场景**


In [ ]:
import numpy as np

# 3 个方程，2 个未知数——通常无精确解
X = np.array([[1.0, 2.0],
              [3.0, 4.0],
              [5.0, 6.0]])  # 3×2
y = np.array([4.0, 10.0, 16.0])

# 伪逆解：最小二乘意义上的最佳近似
w_pinv = np.linalg.pinv(X) @ y
print(f"伪逆解: w = {w_pinv.round(4)}")

# 检查：X @ w 应该"尽可能接近" y，但不一定等于 y
y_pred = X @ w_pinv
print(f"X @ w = {y_pred.round(4)}")
print(f"原始 y = {y}")
print(f"误差 ‖Xw-y‖ = {np.linalg.norm(y_pred - y):.4f} ← 是所有可能的 w 中最小的")

# 验证：如果 X 可逆，pinv == inv
A_square = np.array([[2.0, 1.0], [1.0, 3.0]])
print(f"\n可逆方阵: pinv ≈ inv? {np.allclose(np.linalg.pinv(A_square), np.linalg.inv(A_square))} ✓")


---


### 7.5　AI 应用：线性回归闭式解 —— 一行 NumPy 秒算最优权重

现在到达本章的高潮：把逆矩阵和伪逆应用于 AI 最经典的公式。

线性回归问题：给定 n 个样本 **X**（n×d）和标签 **y**（n,），找权重向量 **w**（d,）使得 **X****w** ≈ **y**。最小二乘的闭式解：

$$**w** = (**X**^T **X**)^{-1} **X**^T **y**$$

这个公式就是伪逆在特定条件下的展开——`pinv(X) = (XᵀX)⁻¹Xᵀ`（当 X 列满秩时）。你不需要 for 循环、不需要梯度下降——一行 NumPy 直接算出最优权重。

📐 **定义　正规方程（Normal Equation）**：**w** = (**X**ᵀ**X**)⁻¹**X**ᵀ**y**。对 XᵀX 求逆即可得到使平方误差最小的权重向量。适用于小数据（n < 10000），大数据时梯度下降（第 12 章）更高效。

💻 **代码　线性回归闭式解：从生成数据到拟合直线**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ===== 生成合成数据：y = 3x - 2 + 噪声 =====
n = 100
X_raw = np.random.uniform(-3, 3, size=n)
true_w, true_b = 3.0, -2.0
y = true_w * X_raw + true_b + np.random.randn(n) * 1.5

# 构建设计矩阵 X：加一列全 1 代表截距项
X = np.column_stack([X_raw, np.ones(n)])  # shape (100, 2)
# X 的第一列 = x 值，第二列 = 1（对应截距 b）

# ===== 闭式解：一行 NumPy =====
w = np.linalg.inv(X.T @ X) @ X.T @ y  # (XᵀX)⁻¹ Xᵀ y

w_slope, w_intercept = w[0], w[1]
print(f"真实参数: w=3.0, b=-2.0")
print(f"闭式解:   w={w_slope:.3f}, b={w_intercept:.3f}")

# ===== 可视化 =====
plt.figure(figsize=(8, 5))
plt.scatter(X_raw, y, alpha=0.5, s=20, label='训练数据')
x_line = np.linspace(-3, 3, 100)
y_line = w_slope * x_line + w_intercept
plt.plot(x_line, y_line, 'r-', linewidth=2, label=f'拟合: y={w_slope:.2f}x+{w_intercept:.2f}')
plt.plot(x_line, true_w * x_line + true_b, 'g--', linewidth=1.5, label='真实: y=3x-2')
plt.xlabel('x'); plt.ylabel('y'); plt.title('线性回归闭式解：一行 NumPy 秒算最优拟合线')
plt.legend(); plt.grid(alpha=0.3); plt.show()


> **关键洞察**：`(XᵀX)⁻¹Xᵀy` —— 四个矩阵操作，不需要学习率、不需要迭代、不需要担心收敛。这就是闭式解的威力。但它的计算复杂度是 O(d³)（d 是特征数），当特征数上万时，求逆的开销太大——此时梯度下降（第 12 章）登场，用迭代逐步逼近最优解。

🔗 **AI 连接**：第 26 章将对线性和梯度下降两种解法做全面对比（耗时、精度、适用场景）。第 24 章的优化器（Adam 等）本质上是用更聪明的迭代策略替代一次性求逆——因为深度神经网络的参数动辄上亿，对亿×亿矩阵求逆在物理上不可能。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）什么样的矩阵不可逆？用"空间压扁"的比喻解释奇异矩阵为什么没有逆。
2. （概念）伪逆 `np.linalg.pinv` 和真逆 `np.linalg.inv` 有什么区别？什么时候该用伪逆？
3. （代码）构造一个 3×3 奇异矩阵，验证其行列式为 0，`inv` 报错，但 `pinv` 能返回结果。用 `A @ pinv(A) @ A` 验证伪逆的性质：结果应接近 A。
4. （代码）为方程组 `3x+2y=12, x−y=1` 用矩阵方式求解，并用 `np.linalg.solve` 交叉验证。
---
> 🔗 **章末钩子**：逆矩阵告诉你"怎么撤销变换"，但有些变换虽然可逆，却在某些方向上拉伸特别多、某些方向上拉伸特别少——这些方向和拉伸倍数，就是一个矩阵的"指纹"。这就是特征值和特征向量。
> 预览：下一章——**特征值与特征向量**，抓住主要矛盾。
